<a href="https://colab.research.google.com/github/ctao-dev/trading-system/blob/initial-changes/Trading_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import polars as pl

# 1. Download 5 years of daily Apple data using yfinance
# yfinance returns a Pandas DataFrame by default
aapl_data = yf.download("AAPL", start="2019-01-01", end="2024-01-01")

# 2. Reset the index so the 'Date' becomes a normal column
aapl_data.reset_index(inplace=True)

# 3. Convert the Pandas DataFrame into a fast Polars DataFrame
df = pl.from_pandas(aapl_data)

# 4. Save it as a highly compressed Parquet file!
df.write_parquet("AAPL_daily.parquet")

print("Data successfully saved to Parquet.")

/tmp/ipykernel_880/3985857586.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  aapl_data = yf.download("AAPL", start="2019-01-01", end="2024-01-01")
[*********************100%***********************]  1 of 1 completed


Data successfully saved to Parquet.


In [ ]:
# Create a class to slowly feed data into the system, bypassing RAM restrictions

class DataHandler:

  # define the batched parquet file, as well as an empty iterator
  def __init__(self, path, batch_size=10):
    self.__reader = pl.scan_parquet(path).collect_batches(chunk_size=batch_size);
    self.__buffer = iter([])

  # used in making the data handler processable by for loops
  def __iter__(self):
    return self

  # check if data empty --> get new batch if possible and insert into iterator
  def __next__(self):
    try:
      return next(self.__buffer)
    except StopIteration:
      try:
        batches = next(self.__reader)
      except StopIteration:
        raise StopIteration
      self.__buffer = batches.iter_rows(named=True)
      return next(self.__buffer)

In [ ]:
test = DataHandler(path="AAPL_daily.parquet")

for row in test:
  print(row)

#print(test)

print("Done")

{"('Date', '')": datetime.datetime(2019, 1, 2, 0, 0), "('Close', 'AAPL')": 37.469200134277344, "('High', 'AAPL')": 37.68986030291286, "('Low', 'AAPL')": 36.593684420142715, "('Open', 'AAPL')": 36.750281492025124, "('Volume', 'AAPL')": 148158800}
{"('Date', '')": datetime.datetime(2019, 1, 3, 0, 0), "('Close', 'AAPL')": 33.73699188232422, "('High', 'AAPL')": 34.57454401761466, "('Low', 'AAPL')": 33.69191057763836, "('Open', 'AAPL')": 34.16169817620422, "('Volume', 'AAPL')": 365248800}
{"('Date', '')": datetime.datetime(2019, 1, 4, 0, 0), "('Close', 'AAPL')": 35.17720031738281, "('High', 'AAPL')": 35.24600976737911, "('Low', 'AAPL')": 34.11899163910133, "('Open', 'AAPL')": 34.292195516680664, "('Volume', 'AAPL')": 234428400}
{"('Date', '')": datetime.datetime(2019, 1, 7, 0, 0), "('Close', 'AAPL')": 35.09890365600586, "('High', 'AAPL')": 35.31244611659193, "('Low', 'AAPL')": 34.617251961933846, "('Open', 'AAPL')": 35.281600249745665, "('Volume', 'AAPL')": 219111200}
{"('Date', '')": datet